# NINA LRGB · Alcor Imaging
A reusable workflow for monochrome **L / R / G / B** FITS exposures. Set the input folder, run the preview, inspect the image and alignment report, then export at native resolution.

Results use only the recorded measurements. The default workflow is an **uncalibrated first pass**, with empirical stellar colour balance. It cannot recover a core saturated in every exposure. Originals are read only; results go into a separate directory.


In [ ]:
from google.colab import drive
from pathlib import Path
import sys, subprocess, json
from IPython.display import display, Image as DisplayImage

drive.mount('/content/drive')
INPUT = Path('/content/drive/MyDrive/your_target')  # CHANGE THIS
TARGET = 'target'                                # CHANGE THIS
OUTPUT = INPUT / 'alcor_output'
WORK = Path('/content/alcor_work')
STACK = 'median'  # mean is available; use clipping only with enough frames
GAINS = None      # or [R_gain, G_gain, B_gain]


## Install and check
The repository is pinned to the version tested for this workflow. Local runtime storage keeps temporary arrays off Drive; final products and reports are saved on Drive.


In [ ]:
REPO = Path('/content/alcor-imaging')
REVISION = 'c91714e'
def run_logged(command, cwd=None):
    with subprocess.Popen(command, cwd=cwd, stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True, bufsize=1) as process:
        for line in process.stdout:
            print(line, end='', flush=True)
        if process.wait():
            raise RuntimeError('Command failed; inspect the output above.')
if not REPO.exists():
    run_logged(['git','clone','https://github.com/AltairSo/alcor-imaging.git',str(REPO)])
run_logged(['git','fetch','origin'], cwd=REPO)
run_logged(['git','checkout',REVISION], cwd=REPO)
run_logged([sys.executable,'-m','pip','install','-q','-e',str(REPO),'pytest'])
run_logged([sys.executable,'-m','pytest','-q'],cwd=REPO)
command = [sys.executable,'-u',str(REPO/'examples/process_nina_lrgb.py'),
           '--input',str(INPUT),'--output',str(OUTPUT),'--work',str(WORK),
           '--target',TARGET,'--stack',STACK]
if GAINS is not None:
    command += ['--gains', *map(str,GAINS)]


## Inventory, alignment, and preview
Header filters are checked against filenames. Inconsistent dimensions/settings, missing channels, and unreliable registration stop processing. No frames are silently dropped. Equal exposures are stacked conventionally; mixed exposures use flux-rate HDR integration.


In [ ]:
run_logged(command + ['--stage','preview'])
display(DisplayImage(filename=str(OUTPUT/f'previews/{TARGET}_natural_preview.png'),width=1100))


In [ ]:
import pandas as pd
inventory = pd.read_csv(OUTPUT/'reports/input_inventory.csv')
display(inventory[['name','filter','exposure','fwhm_native_estimate','ellipticity','saturated_percent']])
display(pd.read_json(OUTPUT/'reports/registration_report.json')[
    ['name','matched_stars','median_residual_native_px','p95_residual_native_px']])
display(DisplayImage(filename=str(OUTPUT/'reports/quality/raw_contact_sheet.jpg'),width=1100))
print((OUTPUT/'reports/render_parameters_bin4.json').read_text())


## Full resolution
Run this after inspecting the preview. It writes linear FITS masters, an RGB-only linear cube, three 16-bit display TIFFs, and PNG previews. The L channel contributes 55% to display luminance; RGB chromatic ratios are preserved. For another target, change the configuration and rerun the preview first.


In [ ]:
run_logged(command + ['--stage','full'])
for profile in ['natural','enhanced','inspection']:
    print(profile)
    display(DisplayImage(filename=str(OUTPUT/f'previews/{TARGET}_{profile}_preview.png'),width=1100))
print('Saved:', OUTPUT)


## Re-render without re-stacking (optional)
Edit `reports/render_parameters_bin4.json` only if you want explicit colour/background/white-point adjustments, then run the cell below. Keep the local work directory. The original linear masters remain unchanged.


In [ ]:
# run_logged(command + ['--stage','render'])


In [ ]:
# Verify formats and save the core, star and corner crops plus a technical report.
run_logged([sys.executable, '-u', str(REPO/'examples/inspect_nina_output.py'),
            '--output', str(OUTPUT), '--work', str(WORK), '--target', TARGET])
for name in ['core_native_100pct.png', 'star_native_100pct.png', 'corners_native_100pct.png']:
    display(DisplayImage(filename=str(OUTPUT/'reports/quality'/name)))


## Recommended: reference-star colour calibration
Run this after the native masters are saved. This generic example uses APASS and a solar white reference. For confirmed equipment, select Gaia and explicit filter/sensor curves as described in the README. The earlier empirical images remain available for comparison.


In [ ]:
# 8. Calibrate using catalogued stars (generic APASS mode)
run_logged(['git','fetch','origin'],cwd=REPO)
run_logged(['git','checkout','613ef68'],cwd=REPO)
colour_command=[sys.executable,'-u',str(REPO/'examples/calibrate_nina_colour.py'),
    '--output',str(OUTPUT),'--work',str(WORK),'--target',TARGET,'--method','apass']
run_logged(colour_command+['--stage','calibrate'])
display(DisplayImage(filename=str(OUTPUT/f'previews/{TARGET}_reference_natural_preview.png'),width=1100))


In [ ]:
# 9. Export reference-calibrated native images
run_logged(colour_command+['--stage','render'])
for profile in ['natural','enhanced']:
    print('REFERENCE-CALIBRATED',profile.upper())
    display(DisplayImage(filename=str(OUTPUT/f'previews/{TARGET}_reference_{profile}_preview.png'),width=1100))
print('Colour report:',OUTPUT/'reports/colour_calibration/calibration.json')


## Observation-only RGB baseline
Reference means catalogue-star calibration, never an external image. Enhanced means a stronger display stretch plus this observation's own luminance. This optional baseline uses RGB only, retains the calibrated gains, softens highlights, and preserves more dark transitions. The native core diagnostic marks near-saturation from the raw frames; it does not repair those pixels.

In [ ]:
run_logged(['git','fetch','origin'],cwd=REPO)
run_logged(['git','checkout','c91714e'],cwd=REPO)
run_logged([sys.executable,'-u',str(REPO/'examples/render_observation_baseline.py'),
    '--output',str(OUTPUT),'--work',str(WORK),'--target',TARGET])
for name in ['baseline_comparison.png','core_comparison.png']:
    display(DisplayImage(filename=str(OUTPUT/'reports/observation_baseline'/name),width=1300))
